# §11.3.4 — 1픽셀 이동에 대한 출력 일관성 측정

> 딥러닝 교재 · 3부 11장 3절 4항 (🐍)
> 선행: §11.3.1(표본화 정리와 접힘) · §11.3.2(세 다운샘플링의 비교) · §11.3.3(접힘 손계산)

## 이 노트북이 답하는 질문

1. **§11.3.1의 접힘 공식이 맞는가?** 스펙트럼에서 $X(\omega/2+\pi)$ 항이 실제로 나타나는가.
2. **§11.3.3의 손계산이 맞는가?** $\omega_0=3\pi/4$가 $\pi/2$로 접히는가.
3. **세 다운샘플링 연산의 이동 취약성은 얼마나 다른가?**
4. **솎기 전 저역통과 하나로 얼마나 개선되는가?** (§11.7.2의 예고)

**예상 실행 시간** CPU 약 20초.
이 실험은 학습이 없다. 구조가 만드는 성질만 본다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. §11.3.3 손계산 재현 — $3\pi/4 \to \pi/2$

In [ ]:
N = 256
n = np.arange(N)
w0 = 3*np.pi/4
x_cos = np.cos(w0 * n)
y_cos = x_cos[::2]                       # s=2 다운샘플링
# 접힌 주파수 추정: FFT 최대 성분
sp = np.abs(np.fft.rfft(y_cos))
w_meas = np.argmax(sp[1:]) + 1
w_meas = w_meas * 2*np.pi / len(y_cos)
print(f"이론 접힘: 2π-3π/2 = π/2 = {np.pi/2:.4f} rad")
print(f"실측 최대 성분:        {w_meas:.4f} rad")

---
## 2. 스펙트럼 접힘 — 넓은 대역 신호에서

저주파 봉우리 하나(ω₁)와 나이퀴스트 한계 π/2를 넘는 봉우리 하나(ω₂)를 가진 신호를 만들고,
(i) 그대로 솎기, (ii) 이동 평균 $[1,2,1]/4$ 저역통과 후 솎기를 비교한다.

In [ ]:
w1, w2 = 0.30*np.pi, 0.80*np.pi          # w2 > π/2 → 접힌다
rngl = np.random.default_rng(SEED)
x = 1.0*np.cos(w1*n + 0.3) + 0.8*np.cos(w2*n + 1.1) + 0.15*rngl.standard_normal(N)

blur121 = np.array([1., 2., 1.]) / 4.
def lp(sig):
    return np.convolve(np.pad(sig, 1, mode='wrap'), blur121, mode='valid')

y_naive = x[::2]
y_blur  = lp(x)[::2]

def spec(sig):
    S = np.abs(np.fft.rfft(sig)) / len(sig)
    w = np.linspace(0, np.pi, len(S))
    return w, S

print("접힘 예측: ω₂=0.80π  →  2π−2·0.80π = 0.40π  (솎은 신호의 축에서)")

---
## 3. 이동 일관성 — 세 다운샘플링 경로

파이프라인: [고정 무작위 conv(k=5) → ReLU → **다운샘플링(2×)**] × 2단.
입력을 $v=0..8$ 순환 이동시키며, 특징의 이동 정렬 거리
$$D(v)=\min_{u}\ \frac{\lVert f(T_v x)-T_u f(x)\rVert}{\lVert f(x)\rVert}$$
를 잰다. 완전 등변이면 $4\mid v$에서 $D=0$이어야 한다.

In [ ]:
def conv_same(sig, w):
    k = len(w); h = k//2
    return np.convolve(np.pad(sig, h, mode='wrap'), w[::-1], mode='valid')

def maxpool2(sig):
    return np.maximum(sig[0::2], sig[1::2])

def avgpool2(sig):
    return 0.5*(sig[0::2] + sig[1::2])

def stride2(sig):
    return sig[0::2]

rk = np.random.default_rng(7)
Wc1 = rk.standard_normal(5) / np.sqrt(5)
Wc2 = rk.standard_normal(5) / np.sqrt(5)

def pipeline(x, down, blur=False):
    a = np.maximum(conv_same(x, Wc1) + 0.1, 0)
    if blur: a = lp(a)
    a = down(a)
    a = np.maximum(conv_same(a, Wc2) + 0.1, 0)
    if blur: a = lp(a)
    a = down(a)
    return a

def align_dist(f_shift, f_ref):
    best = np.inf
    for u in range(len(f_ref)):
        d = np.linalg.norm(f_shift - np.roll(f_ref, u))
        best = min(best, d)
    scale = 0.5*(np.linalg.norm(f_ref) + np.linalg.norm(f_shift))
    return best / (scale + 1e-8)

DOWNS = {lab('최대 풀링', 'max pool'): maxpool2,
         lab('평균 풀링', 'avg pool'): avgpool2,
         lab('스트라이드', 'strided'): stride2}
SH = np.arange(0, 9)
N_SIG = 30 if FAST else 100

def make_natural(rn):
    # 매끄러운 봉우리 + 계단 + 잡음: 자연 신호 흉내
    t = np.linspace(0, 1, N, endpoint=False)
    s = np.zeros(N)
    for _ in range(4):
        c, wdt, a = rn.uniform(0, 1), rn.uniform(0.02, 0.08), rn.uniform(0.5, 1.5)
        s += a*np.exp(-((t - c)/wdt)**2)
    s += (t > rn.uniform(0.3, 0.7)).astype(float) * rn.uniform(0.5, 1.0)
    s += 0.1*rn.standard_normal(N)
    return s

D = {k: np.zeros((N_SIG, len(SH))) for k in DOWNS}
Db = {k: np.zeros((N_SIG, len(SH))) for k in DOWNS}
rs = np.random.default_rng(SEED + 5)
for i in range(N_SIG):
    sig = make_natural(rs)
    for name, down in DOWNS.items():
        ref  = pipeline(sig, down, blur=False)
        refb = pipeline(sig, down, blur=True)
        for vi, v in enumerate(SH):
            D[name][i, vi]  = align_dist(pipeline(np.roll(sig, v), down, blur=False), ref)
            Db[name][i, vi] = align_dist(pipeline(np.roll(sig, v), down, blur=True), refb)
print("측정 완료")
for name in DOWNS:
    print(f"{name:12s} 중앙값 D(v=1) = {np.median(D[name][:,1]):.3f}  →  블러 후 {np.median(Db[name][:,1]):.3f}")

---
## 4. 교재 그림 — fig_11_3_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 신호와 naive 다운샘플
ax = axes[0]
seg = slice(0, 96)
ax.plot(n[seg], x[seg], color=CB[0], lw=0.9, label=lab('원신호', 'original'))
ax.plot(n[seg][::2], y_naive[:48], 'o-', color=CB[4], lw=0.8, ms=2.5,
        label=lab('그대로 솎기', 'naive 2x'))
ax.plot(n[seg][::2], y_blur[:48], 's-', color=CB[5], lw=0.8, ms=2.5,
        label=lab('저역통과 후 솎기', 'blur then 2x'))
ax.set_xlabel(lab('표본 $n$', 'sample $n$')); ax.set_ylabel(lab('진폭', 'amplitude'))
ax.set_title(lab('(a) 고주파 성분이 있는 신호의 2배 솎기', '(a) 2x downsampling'), fontsize=10)
ax.legend(fontsize=8)

# (b) 스펙트럼 접힘
ax = axes[1]
w, S = spec(x); ax.plot(w/np.pi, S, color=CB[0], lw=1.2, label=lab('원신호', 'original'))
w, S = spec(y_naive); ax.plot(w/np.pi, S, color=CB[4], lw=1.2, label=lab('그대로 솎기', 'naive'))
w, S = spec(y_blur); ax.plot(w/np.pi, S, color=CB[5], lw=1.2, ls='--', label=lab('저역통과 후', 'blurred'))
ax.axvline(w2/np.pi, color=CB[0], lw=0.7, ls=':')
ax.axvline((2*np.pi - 2*w2)/np.pi, color=CB[4], lw=0.7, ls=':')
ax.annotate(lab('$\\omega_2$', '$\\omega_2$'), (w2/np.pi, 0.42), fontsize=9)
ax.annotate(lab('접힌 $\\omega_2$', 'folded'), ((2*np.pi-2*w2)/np.pi + 0.02, 0.36), color=CB[4], fontsize=9)
ax.set_xlabel(lab('각주파수 $\\omega/\\pi$ (각 신호의 축)', 'frequency $\\omega/\\pi$'))
ax.set_ylabel(lab('스펙트럼 크기', 'magnitude'))
ax.set_title(lab('(b) 접힘의 확인 — 식 (11.\\,3)', '(b) spectral folding'), fontsize=10)
ax.legend(fontsize=8)

# (c) 이동 일관성
ax = axes[2]
for i, name in enumerate(DOWNS):
    ax.plot(SH, np.median(D[name], axis=0), 'o-', color=CB[i+3], ms=3.5, label=name)
ax.set_xlabel(lab('입력 이동량 $v$ (픽셀)', 'input shift $v$'))
ax.set_ylabel(lab('정렬 거리 $D(v)$', 'alignment distance'))
ax.set_title(lab('(c) 이동 일관성 — 블러 없음', '(c) shift consistency (no blur)'), fontsize=10)
ax.legend(fontsize=8)

# (d) 블러 후
ax = axes[3]
for i, name in enumerate(DOWNS):
    ax.plot(SH, np.median(D[name], axis=0), 'o-', color=CB[i+3], ms=3, alpha=0.25)
    ax.plot(SH, np.median(Db[name], axis=0), 's-', color=CB[i+3], ms=3.5,
            label=name + lab(' + 블러', ' + blur'))
ax.set_xlabel(lab('입력 이동량 $v$ (픽셀)', 'input shift $v$'))
ax.set_ylabel(lab('정렬 거리 $D(v)$', 'alignment distance'))
ax.set_title(lab('(d) 솎기 전 저역통과의 효과 (흐린 선 = 블러 없음)', '(d) with pre-blur'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_11_3_4')
plt.show()

> ### 읽는 법
>
> (a) 솎은 두 신호는 **같은 표본 수**인데, 그대로 솎은 쪽은 원신호에 없던 느린 진동을 만들어 낸다.
> (b) 원신호의 $\omega_2$ 봉우리가 솎은 신호에서 $2\pi-2\omega_2$ 자리로 옮겨 앉았다 — §11.3.1의 둘째 항.
> 저역통과를 먼저 걸면 접힐 재료가 지워져 봉우리가 나타나지 않는다.
> (c) $D$는 총 스트라이드의 배수 $v=0,4,8$에서 정확히 0이다(순환 경계 덕분에 남은 부분 등변성).
> 그 사이에서는 아무 필터 없이 솎는 스트라이드가 가장 취약하고, 상자 저역통과를 겸하는 평균 풀링이 가장 완만하다 — §11.3.2의 표와 일치.
> (d) 같은 연산이라도 **솎기 전 블러 하나**로 취약성이 크게 줄어든다. 분류 정확도 수준의 검증은 §11.7.4에서.

---
## 5. 자기 점검

1. (b)에서 $\omega_1=0.3\pi$ 봉우리는 솎은 뒤 $0.6\pi$에 있다. 왜 두 배가 되었고, 왜 접히지는 않았는가?
2. (c)에서 $v=4$의 $D$가 정확히 0이다. 이 실험의 어떤 설정 덕분인가? 제로 패딩 경계의 실제 이미지에서도 0이겠는가? (§11.1.6)
3. $[1,2,1]/4$ 필터의 주파수 응답 $H(\omega)=\cos^2(\omega/2)$을 유도하고, $\omega=\pi$에서 값이 0임을 확인하라.
4. 최대 풀링을 "조밀한 max → 블러 → 솎기"로 분해하면 (§11.7.2) 어느 곡선에 가까워지겠는가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `w2` | 2절 | 0.80π | 접히는 주파수. π/2 아래로 내리면 접힘이 사라진다 |
| `blur121` | 2절 | [1,2,1]/4 | 더 강한 블러 [1,4,6,4,1]/16으로 바꿔 비교 |
| `N_SIG` | 3절 | 100 | 평균 낼 신호 수 |
| 단 수 | 3절 | 2 | 다운샘플링 단을 3으로 늘리면 취약성이 커진다 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")